# Notebook 01 — VaR Methodology

Compares all five VaR methods using SPY as a running example (5 years of data).
Demonstrates why methods diverge — especially during the COVID-19 crash — and
illustrates VaR's subadditivity failure with a numeric example.

Methods compared:
1. Historical Simulation
2. Parametric Normal
3. Parametric Student-t
4. Filtered Historical Simulation (FHS / Hull-White)
5. Cornish-Fisher Expansion
6. Monte Carlo


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

from src.var_models import (
    historical_var, parametric_var, filtered_historical_simulation_var,
    monte_carlo_var, cornish_fisher_var
)
from src.expected_shortfall import historical_es, parametric_es

print('Libraries loaded.')

In [ ]:
# Download 5 years of SPY
spy = yf.download('SPY', start='2019-01-01', end='2024-01-01', progress=False)['Close']
returns = spy.pct_change().dropna()
print(f'SPY returns: {len(returns)} observations ({returns.index[0].date()} to {returns.index[-1].date()})')
print(f'Annualised vol: {returns.std() * np.sqrt(252):.1%}')
print(f'Skewness:       {returns.skew():.3f}')
print(f'Excess kurtosis:{returns.kurtosis():.3f}')

## 1. Compute all VaR methods at 95% and 99% confidence

In [ ]:
results = {}
for cl, label in [(0.05, '95%'), (0.01, '99%')]:
    mc_var, mc_se = monte_carlo_var(returns, cl, n_simulations=100_000)
    results[label] = {
        'Historical':       historical_var(returns, cl),
        'Parametric Normal':parametric_var(returns, cl, distribution='normal'),
        'Parametric t':     parametric_var(returns, cl, distribution='student_t'),
        'FHS (EWMA)':       filtered_historical_simulation_var(returns, cl),
        'Monte Carlo':      mc_var,
        'Cornish-Fisher':   cornish_fisher_var(returns, cl),
        'ES (Historical)':  historical_es(returns, cl),
        'ES (Parametric)':  parametric_es(returns, cl),
    }

comparison = pd.DataFrame(results)
comparison.index.name = 'Method'
comparison_pct = comparison * 100
print(comparison_pct.round(3).to_string())

## 2. Rolling VaR — Crisis Period Divergence

In [ ]:
# Rolling 63-day (3M) window, 95% VaR for each method
window = 63
rolling_hist = returns.rolling(window).quantile(0.05)
rolling_param = returns.rolling(window).apply(
    lambda r: parametric_var(pd.Series(r), 0.05), raw=False
)

fig = go.Figure()
fig.add_trace(go.Scatter(x=returns.index, y=returns, mode='lines', name='SPY Returns',
                          line=dict(color='lightgrey', width=0.5), opacity=0.6))
fig.add_trace(go.Scatter(x=rolling_hist.index, y=rolling_hist,
                          name='Rolling Historical VaR (3M)', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=rolling_param.index, y=rolling_param,
                          name='Rolling Parametric VaR (3M)', line=dict(color='red', dash='dash')))
fig.update_layout(
    title='Rolling 95% VaR: Historical vs Parametric — SPY 2019-2024',
    xaxis_title='Date', yaxis_title='VaR (return)',
    height=450
)
fig.show()

## 3. Return Distribution with VaR Thresholds

In [ ]:
fig = go.Figure()
fig.add_trace(go.Histogram(x=returns * 100, nbinsx=80, name='SPY Daily Returns (%)',
                            marker_color='steelblue', opacity=0.7))

colors = ['red', 'orange', 'green', 'purple', 'brown']
labels = ['Historical', 'Parametric Normal', 'Parametric t', 'FHS', 'Cornish-Fisher']
for i, (label, col) in enumerate(zip(labels, colors)):
    val = results['95%'][label] * 100
    fig.add_vline(x=val, line_color=col, line_dash='dash',
                  annotation_text=f'{label}: {val:.2f}%', annotation_position='top left')

fig.update_layout(
    title='SPY Return Distribution with 95% VaR Thresholds (5 years)',
    xaxis_title='Daily Return (%)', yaxis_title='Frequency',
    height=450
)
fig.show()

## 4. VaR Subadditivity Failure — Numeric Example

VaR is NOT a coherent risk measure. It fails subadditivity: `VaR(A+B) > VaR(A) + VaR(B)` 
is possible. Expected Shortfall (ES) satisfies subadditivity by construction.


In [ ]:
# Download two assets: SPY and TLT (negatively correlated in most regimes)
data = yf.download(['SPY', 'TLT'], start='2020-01-01', end='2024-01-01', progress=False)['Close']
r = data.pct_change().dropna()

r_spy = r['SPY']
r_tlt = r['TLT']
r_combined = 0.5 * r_spy + 0.5 * r_tlt  # equal-weight portfolio

cl = 0.01  # 99% VaR
var_spy = historical_var(r_spy, cl)
var_tlt = historical_var(r_tlt, cl)
var_combined = historical_var(r_combined, cl)
sum_of_vars = 0.5 * var_spy + 0.5 * var_tlt

es_spy = historical_es(r_spy, cl)
es_tlt = historical_es(r_tlt, cl)
es_combined = historical_es(r_combined, cl)
sum_of_es = 0.5 * es_spy + 0.5 * es_tlt

print('=' * 60)
print('99% VaR SUBADDITIVITY CHECK (2020-2024)')
print('=' * 60)
print(f'VaR(SPY):                   {var_spy*100:.3f}%')
print(f'VaR(TLT):                   {var_tlt*100:.3f}%')
print(f'Weighted sum of VaRs:        {sum_of_vars*100:.3f}%')
print(f'Portfolio VaR (50/50):       {var_combined*100:.3f}%')
print(f'Diversification (VaR):       {(var_combined - sum_of_vars)*100:.3f}%')
print()
print(f'ES(SPY):                     {es_spy*100:.3f}%')
print(f'ES(TLT):                     {es_tlt*100:.3f}%')
print(f'Weighted sum of ES:           {sum_of_es*100:.3f}%')
print(f'Portfolio ES (50/50):         {es_combined*100:.3f}%')
print(f'Diversification (ES):         {(es_combined - sum_of_es)*100:.3f}%')
print()
div_var = var_combined > sum_of_vars
div_es  = es_combined >= sum_of_es
print(f'VaR subadditivity satisfied:  {not div_var}  (expected: usually True but NOT guaranteed)')
print(f'ES  subadditivity satisfied:  {not div_es}  (always True by construction)')

## 5. Summary Table

In [ ]:
summary = comparison_pct.round(3)
summary.columns = ['95% VaR (%)', '99% VaR (%)']
print('\nAll methods — SPY 2019-2024 (returns expressed as %)')
print(summary.to_string())